# Notebook B5: So sánh công bằng, và chi phí ẩn của QAOA

Đi kèm [B-07: Benchmark công bằng và tài liệu đọc thêm](../../docs/B-extensions/07-benchmarking.md).

Đây là một **thí nghiệm đồ chơi** để minh họa *cách đếm chi phí*, không phải bằng chứng về lợi thế của bên nào. Ta lấy các thể hiện Ising ngẫu nhiên $n = 10$ spin, và so sánh simulated annealing (SA) với QAOA mô phỏng chính xác về: xác suất tìm được optimal solution của một lần chạy, thời gian-tới-lời-giải TTS, và chi phí vòng tối ưu tham số của QAOA.

In [1]:
import sys, time
import numpy as np
from scipy.optimize import minimize
sys.path.insert(0, "../../code")
from qco_utils import qaoa_state, qaoa_expectation

n = 10
rng = np.random.default_rng(42)

def random_instance():
    J = np.triu(rng.normal(size=(n, n)), 1)
    J = J + J.T                                   # đối xứng, đường chéo 0
    return J

def energies_table(J):
    # E(s) = sum_{i<j} J_ij s_i s_j với s_j = 1 - 2 x_j, bit j của chỉ số (little-endian)
    idx = np.arange(2**n)
    S = 1 - 2 * ((idx[:, None] >> np.arange(n)) & 1)       # (2^n, n)
    return 0.5 * np.einsum("ki,ij,kj->k", S, J, S)

J = random_instance()
E = energies_table(J)
print("thể hiện mẫu: E nhỏ nhất =", round(E.min(), 3), "| số optimal solution (suy biến spin-flip):", int((np.abs(E - E.min()) < 1e-9).sum()))

thể hiện mẫu: E nhỏ nhất = -12.474 | số optimal solution (suy biến spin-flip): 2


## 1. Simulated annealing: xác suất thành công theo số sweep

SA đơn giản (Metropolis, từng spin, lịch beta tuyến tính), chạy song song nhiều lần bằng numpy. Một "lần chạy" gồm `sweeps` lượt quét, mỗi lượt cập nhật $n$ spin.

In [2]:
def sa_success(J, E_min, sweeps, reads=2000, beta0=0.1, beta1=3.0, seed=0):
    r = np.random.default_rng(seed)
    S = r.choice([-1, 1], size=(reads, n))
    for sw in range(sweeps):
        beta = beta0 + (beta1 - beta0) * sw / max(1, sweeps - 1)
        for i in range(n):
            field = S @ J[:, i]                           # tổng_j J_ij s_j (J_ii = 0)
            dE = -2 * S[:, i] * field
            accept = (dE < 0) | (r.random(reads) < np.exp(-beta * np.clip(dE, 0, None)))
            S[accept, i] *= -1
    En = 0.5 * np.einsum("ki,ij,kj->k", S, J, S)
    return float((En < E_min + 1e-9).mean())

E_min = E.min()
for sweeps in (5, 10, 20, 50, 100):
    print(f"sweeps = {sweeps:3d}: P(một lần chạy ra optimal solution) = {sa_success(J, E_min, sweeps):.3f}")

sweeps =   5: P(một lần chạy ra optimal solution) = 0.235
sweeps =  10: P(một lần chạy ra optimal solution) = 0.277
sweeps =  20: P(một lần chạy ra optimal solution) = 0.284
sweeps =  50: P(một lần chạy ra optimal solution) = 0.284
sweeps = 100: P(một lần chạy ra optimal solution) = 0.278


## 2. QAOA mô phỏng chính xác: xác suất thành công theo depth $p$

Ta tối ưu $2p$ tham số bằng BFGS từ vài điểm khởi đầu ngẫu nhiên, và đếm **số lần gọi hàm mục tiêu** (`nfev`) mà optimizer cần. Trên phần cứng thật, mỗi lần gọi hàm mục tiêu là một đợt đo nhiều shots.

In [3]:
def qaoa_best(E, p, starts=6, seed=0):
    r = np.random.default_rng(seed)
    best = (np.inf, None); total_nfev = 0
    scale = np.abs(E).max()
    for _ in range(starts):
        x0 = np.concatenate([r.uniform(0, 1.0 / scale * 3, p), r.uniform(0, np.pi / 2, p)])
        res = minimize(lambda t: qaoa_expectation(E, t[:p], t[p:]), x0, method="BFGS")
        total_nfev += res.nfev
        if res.fun < best[0]:
            best = (res.fun, res.x)
    x = best[1]
    prob = np.abs(qaoa_state(E, x[:p], x[p:])) ** 2
    return float(prob[np.abs(E - E.min()) < 1e-9].sum()), total_nfev

t0 = time.time()
for p in (1, 2, 3):
    ps, nfev = qaoa_best(E, p)
    print(f"p = {p}: P(thành công) = {ps:.3f} | số lần gọi hàm mục tiêu của optimizer (6 điểm khởi đầu) = {nfev}")
print(f"({time.time() - t0:.0f} giây)")

p = 1: P(thành công) = 0.029 | số lần gọi hàm mục tiêu của optimizer (6 điểm khởi đầu) = 273
p = 2: P(thành công) = 0.072 | số lần gọi hàm mục tiêu của optimizer (6 điểm khởi đầu) = 595


p = 3: P(thành công) = 0.101 | số lần gọi hàm mục tiêu của optimizer (6 điểm khởi đầu) = 1645
(0 giây)


## 3. Thời gian-tới-lời-giải (TTS) và chi phí ẩn

Nếu một lần chạy thành công với xác suất $p_s$, số shots cần để có xác suất thành công $\ge 99\%$ là
$$R_{99} = \left\lceil \frac{\ln(1 - 0.99)}{\ln(1 - p_s)} \right\rceil .$$

Với SA, chi phí là $R_{99} \times \text{sweeps} \times n$ lần cập nhật spin. Với QAOA, có hai phần: (a) $R_{99}$ shots để sampling với tham số đã tối ưu, và (b) **chi phí tối ưu tham số** = số lần gọi hàm mục tiêu × số shots mỗi lần gọi. Phần (b) thường chiếm đa số nhưng hay bị bỏ qua khi báo cáo.

In [4]:
import math
def r99(ps):
    return 1 if ps >= 0.99 else math.ceil(math.log(0.01) / math.log(1 - ps))

SHOTS_PER_EVAL = 1000        # giả định số shots mỗi lần đánh giá hàm mục tiêu trên phần cứng

# 12 thể hiện ngẫu nhiên
instances = [random_instance() for _ in range(12)]
Es = [energies_table(Ji) for Ji in instances]

print("SA (sweeps cố định = 50):")
sa_ps = [sa_success(Ji, Ei.min(), 50, reads=1000, seed=k) for k, (Ji, Ei) in enumerate(zip(instances, Es))]
print(f"  P thành công: trung vị {np.median(sa_ps):.3f}, thấp nhất {min(sa_ps):.3f}, cao nhất {max(sa_ps):.3f}")
print(f"  R99 (trung vị) = {r99(np.median(sa_ps))} lần chạy; chi phí = {r99(np.median(sa_ps)) * 50 * n} lần cập nhật spin")

for p in (1, 2, 3):
    res = [qaoa_best(Ei, p, starts=4, seed=k) for k, Ei in enumerate(Es)]
    ps_list = [r[0] for r in res]; nfev_list = [r[1] for r in res]
    ps_med = float(np.median(ps_list)); nfev_med = float(np.median(nfev_list))
    shots_final = r99(ps_med)
    shots_opt = nfev_med * SHOTS_PER_EVAL
    print(f"QAOA p = {p}: P thành công trung vị {ps_med:.3f} (thấp nhất {min(ps_list):.3f}, cao nhất {max(ps_list):.3f})")
    print(f"           shots sampling cuối R99 = {shots_final:6d} | shots cho vòng tối ưu ~ {int(shots_opt):9d} ({nfev_med:.0f} lần gọi x {SHOTS_PER_EVAL})")

SA (sweeps cố định = 50):
  P thành công: trung vị 0.757, thấp nhất 0.350, cao nhất 0.990
  R99 (trung vị) = 4 lần chạy; chi phí = 2000 lần cập nhật spin


QAOA p = 1: P thành công trung vị 0.034 (thấp nhất 0.021, cao nhất 0.048)
           shots sampling cuối R99 =    135 | shots cho vòng tối ưu ~    160500 (160 lần gọi x 1000)


QAOA p = 2: P thành công trung vị 0.074 (thấp nhất 0.035, cao nhất 0.135)
           shots sampling cuối R99 =     60 | shots cho vòng tối ưu ~    495000 (495 lần gọi x 1000)


QAOA p = 3: P thành công trung vị 0.116 (thấp nhất 0.075, cao nhất 0.192)
           shots sampling cuối R99 =     38 | shots cho vòng tối ưu ~   1256500 (1256 lần gọi x 1000)


## 4. Đọc kết quả thế nào cho đúng

- Chi phí **vòng tối ưu tham số** của QAOA (hàng trăm nghìn shots ở trên) lớn hơn rất nhiều so với $R_{99}$ shots sampling cuối. Nếu chỉ tính $R_{99}$ thì sẽ đánh giá thấp chi phí thực. Ngược lại, SA không có vòng ngoài này.
- Đơn vị chi phí không so sánh trực tiếp được: "lần cập nhật spin" trên CPU và "shot" trên QPU có thời gian thực khác nhau nhiều bậc. Muốn so sánh nghiêm túc phải dùng **thời gian thực end-to-end** trên phần cứng cụ thể, gồm cả biên dịch, hàng đợi, chuyển dữ liệu.
- Đây là bài $n = 10$ rất nhỏ nên SA chỉ cần vài sweep đã có xác suất thành công đáng kể; không thể suy ra gì về bài lớn. Chỉ có phần *cách đếm* là đáng mang đi.
- Số lần gọi hàm mục tiêu (`nfev`) phụ thuộc optimizer: BFGS ước lượng gradient bằng sai phân hữu hạn nên tốn $2p+1$ lần gọi cho mỗi bước. Optimizer không dùng gradient (như COBYLA) hay kỹ thuật khởi tạo tham số thông minh sẽ cho số khác, nhưng thường vẫn ở cỡ hàng trăm đến hàng nghìn lần gọi. Cả SA lẫn QAOA ở đây đều chưa được tinh chỉnh.
- Các nguyên tắc benchmark đầy đủ nằm ở [B-07](../../docs/B-extensions/07-benchmarking.md).